# Predict which customers will leave(churn)

[Data Source](https://github.com/IBM/telco-customer-churn-on-icp4d/blob/master/data/Telco-Customer-Churn.csv#L7044)

In [101]:
import tensorflow as tf 
import numpy as np 
from sklearn.model_selection import train_test_split 
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

### **Loading data**

In [102]:
df=pd.read_csv('data\Telco-Customer-Churn.csv')

In [103]:
df.head(5)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [104]:
rows,columns =df.shape
print(f"{rows} and {columns} columns")

7043 and 21 columns


In [105]:
df.dtypes

customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

In [106]:
df.isnull().sum()/len(df)*100

customerID          0.0
gender              0.0
SeniorCitizen       0.0
Partner             0.0
Dependents          0.0
tenure              0.0
PhoneService        0.0
MultipleLines       0.0
InternetService     0.0
OnlineSecurity      0.0
OnlineBackup        0.0
DeviceProtection    0.0
TechSupport         0.0
StreamingTV         0.0
StreamingMovies     0.0
Contract            0.0
PaperlessBilling    0.0
PaymentMethod       0.0
MonthlyCharges      0.0
TotalCharges        0.0
Churn               0.0
dtype: float64

In [107]:
values = df["TotalCharges"].unique() 
for index, value in enumerate(values):
    if value == ' ':
        print(index)
        break 
    print(f"{index}: VALUE: {value}: converted values => {float(value)}")

0: VALUE: 29.85: converted values => 29.85
1: VALUE: 1889.5: converted values => 1889.5
2: VALUE: 108.15: converted values => 108.15
3: VALUE: 1840.75: converted values => 1840.75
4: VALUE: 151.65: converted values => 151.65
5: VALUE: 820.5: converted values => 820.5
6: VALUE: 1949.4: converted values => 1949.4
7: VALUE: 301.9: converted values => 301.9
8: VALUE: 3046.05: converted values => 3046.05
9: VALUE: 3487.95: converted values => 3487.95
10: VALUE: 587.45: converted values => 587.45
11: VALUE: 326.8: converted values => 326.8
12: VALUE: 5681.1: converted values => 5681.1
13: VALUE: 5036.3: converted values => 5036.3
14: VALUE: 2686.05: converted values => 2686.05
15: VALUE: 7895.15: converted values => 7895.15
16: VALUE: 1022.95: converted values => 1022.95
17: VALUE: 7382.25: converted values => 7382.25
18: VALUE: 528.35: converted values => 528.35
19: VALUE: 1862.9: converted values => 1862.9
20: VALUE: 39.65: converted values => 39.65
21: VALUE: 202.25: converted values => 2

#### **Descent cleaning**

In [108]:
df_2 = df

In [109]:
df_2["TotalCharges"] = pd.to_numeric(df_2["TotalCharges"], errors="coerce") # coerce basically enable the alg. to be able to replace what cannot be changed into numerical with NaN (not a number)

In [110]:
df_2.iloc[470: 500][["customerID","TotalCharges"]]

,customerID,TotalCharges
470,4933-IKULF,330.60
471,3583-EKAPL,55.00
472,1304-BCCFO,564.40
473,4104-PVRPS,1315.35
474,9399-APLBT,74.70
475,2359-KMGLI,1861.50
476,3780-DDGSE,2747.20
477,4431-EDMIQ,554.05
478,0306-JAELE,453.40
479,6227-HWPWX,994.80


In [111]:
df_2.iloc[488]

customerID                         4472-LVYGI
gender                                 Female
SeniorCitizen                               0
Partner                                   Yes
Dependents                                Yes
tenure                                      0
PhoneService                               No
MultipleLines                No phone service
InternetService                           DSL
OnlineSecurity                            Yes
OnlineBackup                               No
DeviceProtection                          Yes
TechSupport                               Yes
StreamingTV                               Yes
StreamingMovies                            No
Contract                             Two year
PaperlessBilling                          Yes
PaymentMethod       Bank transfer (automatic)
MonthlyCharges                          52.55
TotalCharges                              NaN
Churn                                      No
Name: 488, dtype: object

In [112]:
df_2 = df_2.drop(columns =["customerID"]) # we drop the customer id

In [113]:
df_2.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [114]:
df.isna().sum()

customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64

In [115]:
df_2 = df_2.dropna() # we drop the rows with NaN values

In [116]:
df_2.dtypes

gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges        float64
Churn                   str
dtype: object

In [117]:
df_2=pd.get_dummies(df_2,drop_first=True) # we convert the categorical variables into numerical variables

In [118]:
df_2.dtypes

SeniorCitizen                              int64
tenure                                     int64
MonthlyCharges                           float64
TotalCharges                             float64
gender_Male                                 bool
Partner_Yes                                 bool
Dependents_Yes                              bool
PhoneService_Yes                            bool
MultipleLines_No phone service              bool
MultipleLines_Yes                           bool
InternetService_Fiber optic                 bool
InternetService_No                          bool
OnlineSecurity_No internet service          bool
OnlineSecurity_Yes                          bool
OnlineBackup_No internet service            bool
OnlineBackup_Yes                            bool
DeviceProtection_No internet service        bool
DeviceProtection_Yes                        bool
TechSupport_No internet service             bool
TechSupport_Yes                             bool
StreamingTV_No inter

In [119]:
df_2["Churn_Yes"].value_counts()

Churn_Yes
False    5163
True     1869
Name: count, dtype: int64

In [120]:
df_2["Churn"]= (df_2["Churn_Yes"] == 1).astype(int) # we create a new column called churn which is equal to 1 if the customer has churned and 0 if the customer has not churned

In [121]:
df_2["Churn"].value_counts()

Churn
0    5163
1    1869
Name: count, dtype: int64

In [122]:
df_2["Churn_Yes"].value_counts()

Churn_Yes
False    5163
True     1869
Name: count, dtype: int64

In [123]:
X =df_2.drop(columns=["Churn", "Churn_Yes"]).values.astype("float32") # we drop the churn column from the dataset and store it in X
y = df_2["Churn"].values
X, y

(array([[  0.  ,   1.  ,  29.85, ...,   0.  ,   1.  ,   0.  ],
        [  0.  ,  34.  ,  56.95, ...,   0.  ,   0.  ,   1.  ],
        [  0.  ,   2.  ,  53.85, ...,   0.  ,   0.  ,   1.  ],
        ...,
        [  0.  ,  11.  ,  29.6 , ...,   0.  ,   1.  ,   0.  ],
        [  1.  ,   4.  ,  74.4 , ...,   0.  ,   0.  ,   1.  ],
        [  0.  ,  66.  , 105.65, ...,   0.  ,   0.  ,   0.  ]],
       shape=(7032, 30), dtype=float32),
 array([0, 0, 1, ..., 0, 1, 0], shape=(7032,)))

#### **Split then scale the data**

In [124]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.3,stratify=y,random_state=42)

In [125]:
scaler = StandardScaler() # we create an instance of the StandardScaler class
scaler.fit(X_train) # we fit the scaler to the training data
X_train,X_test = scaler.transform(X_train), scaler.transform(X_test)# we transform the training data

In [126]:
X_train

array([[-0.4405457 , -0.833469  ,  0.44474947, ..., -0.5276756 ,
         1.3986266 , -0.5367909 ],
       [-0.4405457 , -0.508058  , -1.4921349 , ..., -0.5276756 ,
        -0.71498716,  1.8629229 ],
       [-0.4405457 , -1.2402327 , -0.12045064, ...,  1.8951037 ,
        -0.71498716, -0.5367909 ],
       ...,
       [-0.4405457 ,  0.14276391,  0.92731786, ...,  1.8951037 ,
        -0.71498716, -0.5367909 ],
       [-0.4405457 , -0.91482174,  0.03489676, ..., -0.5276756 ,
        -0.71498716,  1.8629229 ],
       [-0.4405457 , -1.2809091 , -1.3235666 , ..., -0.5276756 ,
        -0.71498716, -0.5367909 ]], shape=(4922, 30), dtype=float32)

## **Modelling**

#### **Logistic regression**

In [127]:
df_2.columns

Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges',
       'gender_Male', 'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes',
       'MultipleLines_No phone service', 'MultipleLines_Yes',
       'InternetService_Fiber optic', 'InternetService_No',
       'OnlineSecurity_No internet service', 'OnlineSecurity_Yes',
       'OnlineBackup_No internet service', 'OnlineBackup_Yes',
       'DeviceProtection_No internet service', 'DeviceProtection_Yes',
       'TechSupport_No internet service', 'TechSupport_Yes',
       'StreamingTV_No internet service', 'StreamingTV_Yes',
       'StreamingMovies_No internet service', 'StreamingMovies_Yes',
       'Contract_One year', 'Contract_Two year', 'PaperlessBilling_Yes',
       'PaymentMethod_Credit card (automatic)',
       'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check',
       'Churn_Yes', 'Churn'],
      dtype='str')

In [141]:
X =df_2.drop(columns=["Churn", "Churn_Yes"]).values.astype("float32") 
y = df_2["Churn"]


In [147]:
df_2.shape

(7032, 32)

In [142]:
X_train_log, X_test_log, y_train_log, y_test_log = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [143]:
log_reg_model = LogisticRegression(max_iter=1000)
log_reg_model.fit(X_train_log, y_train_log)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use i

In [144]:
y_pred_class = log_reg_model.predict(X_test_log)
y_pred_proba = log_reg_model.predict_proba(X_test_log)

In [133]:
model = tf.keras.models.Sequential([
  tf.keras.layers.Input((X_train.shape[1],)), #  data is already flat hence we do not need to flatten the features trsnalte into an input

  # This is your first hidden layer n 128 neurons
  tf.keras.layers.Dense(32, activation='relu'),
  tf.keras.layers.Dropout(0.3),# 20% of your data is left out to lower the chance pof overfittinhs

  # This is your first hidden layer n 64 neurons
  tf.keras.layers.Dense(16, activation='relu'),
  tf.keras.layers.Dropout(0.3),# 20% of your data is left out to lower the chance pof overfittinh
  
  tf.keras.layers.Dense(10, activation='softmax')
])

In [134]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [135]:
model.fit(X_train, y_train, epochs=30)

Epoch 1/30
154/154 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.5719 - loss: 1.3707
Epoch 2/30
154/154 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7412 - loss: 0.6343
Epoch 3/30
154/154 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7597 - loss: 0.5256
Epoch 4/30
154/154 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7668 - loss: 0.4956
Epoch 5/30
154/154 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.7790 - loss: 0.4731
Epoch 6/30
154/154 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.7810 - loss: 0.4671
Epoch 7/30
154/154 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7806 - loss: 0.4646
Epoch 8/30
154/154 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7816 - loss: 0.4589
Epoch 9/30
154/154 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7830 - loss: 0.4550
Epoch 10/30
154/154 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7920 - loss: 0.4444
Epoch 11/30
154/154 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7911 - loss: 0.4484
Epoch 12/30
154/154 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step

In [136]:
model_two = tf.keras.models.Sequential([
  tf.keras.layers.Input((X_train.shape[1],)), #  data is already flat hence we do not need to flatten the features trsnalte into an input

  # This is your first hidden layer n 32 neurons
  tf.keras.layers.Dense(32, activation='relu'),
  tf.keras.layers.Dropout(0.3),# 20% of your data is left out to lower the chance pof overfittinhs

  # This is your first hidden layer n 16 neurons
  tf.keras.layers.Dense(16, activation='relu'),
  tf.keras.layers.Dropout(0.3),# 20% of your data is left out to lower the chance pof overfittinh
  
  tf.keras.layers.Dense(1, activation='sigmoid')
])

In [137]:

model_two.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy'])

In [138]:
model_two.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy'])

In [139]:
model_two.fit(X_train, y_train, epochs=30, batch_size=32)

Epoch 1/30
154/154 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.7235 - loss: 0.5306
Epoch 2/30
154/154 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7649 - loss: 0.4728
Epoch 3/30
154/154 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7765 - loss: 0.4594
Epoch 4/30
154/154 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7832 - loss: 0.4496
Epoch 5/30
154/154 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7836 - loss: 0.4398
Epoch 6/30
154/154 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7897 - loss: 0.4370
Epoch 7/30
154/154 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7922 - loss: 0.4314
Epoch 8/30
154/154 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7944 - loss: 0.4343
Epoch 9/30
154/154 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7893 - loss: 0.4323
Epoch 10/30
154/154 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7999 - loss: 0.4312
Epoch 11/30
154/154 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7978 - loss: 0.4281
Epoch 12/30
154/154 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step

### **Evaluation**

In [150]:
print(classification_report(y_test_log, y_pred_class))

              precision    recall  f1-score   support

           0       0.84      0.90      0.87      1033
           1       0.64      0.51      0.57       374

    accuracy                           0.79      1407
   macro avg       0.74      0.70      0.72      1407
weighted avg       0.78      0.79      0.79      1407

